In [1]:

import copy
import os
import sys
from pathlib import Path

os.environ.setdefault('MPLCONFIGDIR', '/tmp/matplotlib')

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from sklearn.metrics import average_precision_score, precision_recall_curve, auc
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

PROJECT_ROOT = Path('/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from DMTimeShardDataset import DMTimeShardDataset
from moe.train_joint_ensemble import build_joint_model
from training_utils import label_encoding

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [ ]:

# MOE_CHECKPOINT = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe/moe_runs/randomsearch_new_routingloss/joint_moe_worker0_trial0_seed42_expertlr8p285em05_rejectorlr1p122em06_wd1em05_temp1_topknoisestd0p5_noisestart0_noiseep0_aux0p8_auxep0_budget3_routing2_auxwarmup1/checkpoints/joint_cascade_moe_best.pth")

# MOE_CHECKPOINT = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/moe/moe_runs/randomsearch_new_budgetloss_expertanalysis_2/50_50_reject/checkpoints/joint_cascade_moe_best.pth")

MOE_CHECKPOINT = Path("/cephfs/users/oleksjuk/MA/WP2-1/single_pulse_classifier_training/final_checkpoints/joint_cascade_moe_best.pth")

# DATASET_CFG = {
#     'output_dir': '/raid/outputs',
#     'prefix': 'B0531+21_59000_48386',
# }


DATASET_CFG = {
    'output_dir': '/cephfs/users/oleksjuk/MA/WP2-1/DM_time_dataset_creator/outputs',
    'prefix': 'B0531+21_59000_48386',
}

TARGET_R1_REJECT_RATE = 0.30
TARGET_R2_REJECT_RATE_LOCAL = 0.30
BATCH_SIZE = 1024
NUM_WORKERS = 10



In [3]:

ckpt = torch.load(MOE_CHECKPOINT, map_location='cpu')
config = copy.deepcopy(ckpt['config'])
config['dataset']['output_dir'] = DATASET_CFG['output_dir']
config['dataset']['prefix'] = DATASET_CFG['prefix']

model = build_joint_model(config, DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

small_model = model.f_small
mid_model = model.f_mid
large_model = model.f_large
r1_model = model.r1
r2_model = model.r2

for module in (small_model, mid_model, large_model, r1_model, r2_model):
    module.eval()

print(f"Loaded MoE checkpoint: {MOE_CHECKPOINT.name}")
print(f"Best epoch: {ckpt.get('epoch')}")
print(f"Validation top-k accuracy in checkpoint: {ckpt.get('metrics', {}).get('val_topk/accuracy', float('nan')):.4f}")
print(f"R1 model: {r1_model.__class__.__name__}")
print(f"R2 model: {r2_model.__class__.__name__}")


Loaded MoE checkpoint: joint_cascade_moe_best.pth
Best epoch: 75
Validation top-k accuracy in checkpoint: 0.8725
R1 model: ConvMLPEmbeddingProcessing
R2 model: ConvMLPEmbeddingProcessing


In [ ]:
datasets = {}
loaders = {}
for split in ('val', 'test'):
    dataset = DMTimeShardDataset(DATASET_CFG, use_freq_time=True, split=split)
    dataset.labels = label_encoding(dataset.labels.astype(object))
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    datasets[split] = dataset
    loaders[split] = loader
    print(f'{split} samples: {len(dataset)}')


In [ ]:
def collect_moe_scores(loader, split):
    all_r1_scores = []
    all_r2_scores = []
    all_small_preds = []
    all_mid_preds = []
    all_large_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in tqdm(loader, desc=f'Collecting MoE {split} scores'):
            labels = batch['label'].to(DEVICE)
            outputs = model._forward_all_aux(batch)

            expert_logits = outputs['expert_logits']
            all_r1_scores.append(outputs['rejector_probs']['r1'].detach().cpu())
            all_r2_scores.append(outputs['rejector_probs']['r2'].detach().cpu())
            all_small_preds.append(expert_logits[:, 0].argmax(dim=1).detach().cpu())
            all_mid_preds.append(expert_logits[:, 1].argmax(dim=1).detach().cpu())
            all_large_preds.append(expert_logits[:, 2].argmax(dim=1).detach().cpu())
            all_labels.append(labels.detach().cpu())

    arrays = {
        'r1_scores': torch.cat(all_r1_scores).numpy(),
        'r2_scores': torch.cat(all_r2_scores).numpy(),
        'small_preds': torch.cat(all_small_preds).numpy(),
        'mid_preds': torch.cat(all_mid_preds).numpy(),
        'large_preds': torch.cat(all_large_preds).numpy(),
        'labels': torch.cat(all_labels).numpy(),
    }
    print(f"Collected scores for {len(arrays['labels'])} {split} samples")
    print(f"{split} f_small accuracy: {np.mean(arrays['small_preds'] == arrays['labels']):.4f}")
    print(f"{split} f_mid accuracy:   {np.mean(arrays['mid_preds'] == arrays['labels']):.4f}")
    print(f"{split} f_large accuracy: {np.mean(arrays['large_preds'] == arrays['labels']):.4f}")
    return arrays

scores_by_split = {split: collect_moe_scores(loaders[split], split) for split in ('val', 'test')}

val_arrays = scores_by_split['val']
test_arrays = scores_by_split['test']

# Keep the old variable names for the plotting cells below. They now refer to test,
# while thresholds are calibrated on val in the next cell.
r1_scores_np = test_arrays['r1_scores']
r2_scores_np = test_arrays['r2_scores']
small_preds_np = test_arrays['small_preds']
mid_preds_np = test_arrays['mid_preds']
large_preds_np = test_arrays['large_preds']
labels_np = test_arrays['labels']


In [ ]:
def threshold_for_reject_rate(scores, target_reject_rate):
    scores = np.asarray(scores, dtype=float).reshape(-1)
    if scores.size == 0:
        raise ValueError('scores must not be empty')
    if not 0.0 <= target_reject_rate <= 1.0:
        raise ValueError('target_reject_rate must be in [0, 1]')

    k = int(np.rint(target_reject_rate * scores.size))
    k = int(np.clip(k, 0, scores.size))
    if k == 0:
        threshold = float('inf')
    elif k == scores.size:
        threshold = float('-inf')
    else:
        threshold = float(np.sort(scores)[::-1][k - 1])

    mask = scores >= threshold
    return {
        'threshold': threshold,
        'target_reject_rate': float(target_reject_rate),
        'achieved_reject_rate': float(mask.mean()),
        'n_samples': int(scores.size),
        'n_rejected': int(mask.sum()),
        'mask': mask,
    }


def print_rejector_pr(name, y_true_reject, scores):
    precision, recall, _ = precision_recall_curve(y_true_reject, scores)
    ap = average_precision_score(y_true_reject, scores)
    pr_auc_trapz = auc(recall, precision)
    print(f'{name} routing target positive rate: {np.mean(y_true_reject):.4f}')
    print(f'{name} Average Precision:            {ap:.4f}')
    print(f'{name} trapezoidal PR-AUC:           {pr_auc_trapz:.4f}')
    return precision, recall, ap


def evaluate_thresholds(arrays, r1_threshold, r2_threshold, split):
    r1_reject = arrays['r1_scores'] >= r1_threshold
    r2_reject = r1_reject & (arrays['r2_scores'] >= r2_threshold)
    r2_accept = r1_reject & ~r2_reject

    y_pred = arrays['small_preds'].copy()
    y_pred[r2_accept] = arrays['mid_preds'][r2_accept]
    y_pred[r2_reject] = arrays['large_preds'][r2_reject]

    return {
        'split': split,
        'r1_threshold': float(r1_threshold),
        'r1_reject_rate': float(r1_reject.mean()),
        'r2_threshold': float(r2_threshold),
        'r2_local_reject_rate': float((arrays['r2_scores'][r1_reject] >= r2_threshold).mean()),
        'global_large_rate': float(r2_reject.mean()),
        'global_mid_or_large_rate': float(r1_reject.mean()),
        'ensemble_acc': float((y_pred == arrays['labels']).mean()),
        'n_samples': int(arrays['labels'].size),
        'n_r1_rejected': int(r1_reject.sum()),
        'n_r2_rejected': int(r2_reject.sum()),
    }


In [ ]:
# Calibrate thresholds on validation only.
r1_selection_val = threshold_for_reject_rate(val_arrays['r1_scores'], TARGET_R1_REJECT_RATE)
r1_threshold = r1_selection_val['threshold']
r1_mask_val = r1_selection_val['mask']

r2_active_scores_val = val_arrays['r2_scores'][r1_mask_val]
r2_selection_val = threshold_for_reject_rate(r2_active_scores_val, TARGET_R2_REJECT_RATE_LOCAL)
r2_threshold = r2_selection_val['threshold']

# Apply the validation thresholds unchanged to test. This is the r2 subset that
# actually reaches r2 in the final test evaluation.
r1_mask_np = r1_scores_np >= r1_threshold
r2_active_scores = r2_scores_np[r1_mask_np]

val_operating_point = evaluate_thresholds(val_arrays, r1_threshold, r2_threshold, 'val')
test_operating_point = evaluate_thresholds(test_arrays, r1_threshold, r2_threshold, 'test')
selected_df = pd.DataFrame([val_operating_point, test_operating_point])

print('Selected thresholds from validation')
print(f"  R1 target reject rate on val:      {TARGET_R1_REJECT_RATE:.4f}")
print(f"  R1 achieved reject rate on val:    {r1_selection_val['achieved_reject_rate']:.4f} ({r1_selection_val['n_rejected']}/{r1_selection_val['n_samples']})")
print(f"  R1 threshold:                      {r1_threshold:.6f}")
print()
print(f"  R2 target local reject rate on val:{TARGET_R2_REJECT_RATE_LOCAL:.4f}")
print(f"  R2 achieved local rate on val:     {r2_selection_val['achieved_reject_rate']:.4f} ({r2_selection_val['n_rejected']}/{r2_selection_val['n_samples']})")
print(f"  R2 threshold:                      {r2_threshold:.6f}")
print()
print('Full cascade at validation-calibrated thresholds')
display(selected_df)



# PR curves

Rejector targets use `current_wrong`: r1 positive means `f_small` is wrong; r2 positive means `f_mid` is wrong on the samples forwarded by r1. Thresholds are calibrated on validation and then applied unchanged to test.


In [ ]:
from matplotlib.ticker import FuncFormatter

r1_targets = (small_preds_np != labels_np).astype(int)

r2_targets = (
    mid_preds_np[r1_mask_np] != labels_np[r1_mask_np]
).astype(int)

r1_precision, r1_recall, r1_ap = print_rejector_pr(
    "R1",
    r1_targets,
    r1_scores_np,
)

r2_precision, r2_recall, r2_ap = print_rejector_pr(
    "R2",
    r2_targets,
    r2_active_scores,
)

r1_baseline = r1_targets.mean()
r2_baseline = r2_targets.mean()
r2_ap_display = np.ceil(r2_ap * 1000) / 1000

comma_formatter = FuncFormatter(
    lambda value, pos: f"{value:.1f}".replace(".", ",")
)

fig, axes = plt.subplots(
    1,
    2,
    figsize=(6.8, 2.3),
)

# ---------------------------------------------------------
# r1
# ---------------------------------------------------------
axes[0].plot(
    r1_recall,
    r1_precision,
    linewidth=1.0,
    label=f"AP = {r1_ap:.3f}".replace(".", ","),
)

axes[0].axhline(
    r1_baseline,
    linestyle="--",
    color="0.45",
    linewidth=1.0,
    label=f"Baseline = {r1_baseline:.3f}".replace(".", ","),
)

axes[0].set_title(
    r"$r_1$",
    pad=3,
)

axes[0].set_xlabel(
    "Recall",
    labelpad=2,
)

axes[0].set_ylabel(
    "Precision",
    labelpad=2,
)

# ---------------------------------------------------------
# r2
# ---------------------------------------------------------
axes[1].plot(
    r2_recall,
    r2_precision,
    linewidth=1.0,
    label=f"AP = {r2_ap_display:.3f}".replace(".", ","),
)

axes[1].axhline(
    r2_baseline,
    linestyle="--",
    color="0.45",
    linewidth=1.0,
    label=f"Baseline = {r2_baseline:.3f}".replace(".", ","),
)

axes[1].set_title(
    r"$r_2$",
    pad=3,
)

axes[1].set_xlabel(
    "Recall",
    labelpad=2,
)

axes[1].set_ylabel(
    "Precision",
    labelpad=2,
)

# ---------------------------------------------------------
# Gemeinsamer Stil
# ---------------------------------------------------------
for ax in axes:
    ax.set_xlim(0.0, 1.0)
    ax.set_ylim(0.0, 1.0)

    ax.xaxis.set_major_formatter(comma_formatter)
    ax.yaxis.set_major_formatter(comma_formatter)

    ax.grid(
        axis="both",
        color="0.90",
        linewidth=0.7,
        linestyle="-",
    )

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    ax.legend(
        loc="lower left",
        frameon=False,
        fontsize=7,
    )

fig.subplots_adjust(
    left=0.075,
    right=0.995,
    top=0.91,
    bottom=0.20,
    wspace=0.22,
)

pdf_path = "rejector_precision_recall.pdf"

fig.savefig(
    pdf_path,
)

plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

r1_threshold_grid = np.sort(np.unique(val_arrays['r1_scores']))
r1_rate_grid = np.array([(val_arrays['r1_scores'] >= t).mean() for t in r1_threshold_grid])
axes[0].plot(r1_threshold_grid, r1_rate_grid, linewidth=1)
axes[0].axhline(val_operating_point['r1_reject_rate'], linestyle='--', label=f"val rate={val_operating_point['r1_reject_rate']:.3f}")
axes[0].axhline(test_operating_point['r1_reject_rate'], linestyle=':', label=f"test rate={test_operating_point['r1_reject_rate']:.3f}")
axes[0].axvline(r1_threshold, linestyle='--', label=f't={r1_threshold:.4f}')
axes[0].set_xlabel('R1 threshold')
axes[0].set_ylabel('Reject rate')
axes[0].set_title('R1 reject rate over validation threshold')
axes[0].grid(True)
axes[0].legend()

r2_threshold_grid = np.sort(np.unique(r2_active_scores_val))
r2_rate_grid = np.array([(r2_active_scores_val >= t).mean() for t in r2_threshold_grid])
axes[1].plot(r2_threshold_grid, r2_rate_grid, linewidth=1)
axes[1].axhline(val_operating_point['r2_local_reject_rate'], linestyle='--', label=f"val rate={val_operating_point['r2_local_reject_rate']:.3f}")
axes[1].axhline(test_operating_point['r2_local_reject_rate'], linestyle=':', label=f"test rate={test_operating_point['r2_local_reject_rate']:.3f}")
axes[1].axvline(r2_threshold, linestyle='--', label=f't={r2_threshold:.4f}')
axes[1].set_xlabel('R2 threshold')
axes[1].set_ylabel('Local reject rate among R1-routed samples')
axes[1].set_title('R2 reject rate over validation threshold')
axes[1].grid(True)
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:

output_dir = PROJECT_ROOT / 'moe' / 'artifacts' / 'operating_points'
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / 'moe_selected_r1_r2_thresholds_val.csv'
selected_df.to_csv(output_file, index=False)
print(f'Saved selected thresholds to {output_file}')
